# 01. Compaction 与摘要

对话变长后只有三条路：全量保留（贵）、直接截断（失忆）、滚动摘要（有损压缩）。
Compaction 的本质是**信息取舍**：决策、约束、事实必须保留；寒暄、试错过程可以丢弃。

> **检查点**：能说出「摘要写得太短」和「摘要写得太长」各自的失败模式。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()

🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'context-engineering' / 'scripts' / 'compaction_demo.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--strategy', 'all'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

Question: Final question: given all constraints, can we run a 15W payload continuously? Why or why not?
Critical fact location: turn 1 of 9 ('40W power budget')

=== Strategy: full (toy) ===
  Context tokens: 344  | early fact present: True
  Answer: No. A 15W continuous payload against the 40W total budget leaves only 25W for OBC (5-7W), radio (8W transmit), heaters, and battery charging — with no margin for eclipse. Duty-cycling the payload (e.g., 50%) would fit.

=== Strategy: window (toy) ===
  Context tokens: 157  | early fact present: False
  Answer: Insufficient context: I don't have the power budget figure in what I can see, so I can't determine whether 15W continuous fits. (This is the failure mode of naive truncation.)

=== Strategy: summary (toy) ===
  Context tokens: 198  | early fact present: True
  Answer: No. A 15W continuous payload against the 40W total budget leaves only 25W for OBC (5-7W), radio (8W transmit), heaters, and battery charging — with no margin for eclips

## 三种策略对比

| 策略 | Token 成本 | 早期事实 | 适用场景 |
|------|-----------|---------|---------|
| 全量 (full) | 线性增长，无上限 | 完整保留 | 短对话、审计场景 |
| 滑动窗口 (window) | 恒定最低 | 丢失（上面的 window 回答已暴露） | 闲聊、无状态任务 |
| 滚动摘要 (summary) | ~1/3 全量 | 关键约束保留 | 长任务、工程对话 |

摘要失败模式：太短 → 约束丢失（agent 违反 40W 预算）；太长 → 退化为全量，失去压缩意义。

In [3]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'context-engineering' / 'scripts' / 'compaction_demo.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--strategy', 'all', '--use-api'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

Question: Final question: given all constraints, can we run a 15W payload continuously? Why or why not?
Critical fact location: turn 1 of 9 ('40W power budget')

=== Strategy: full (API) ===
  Context tokens: 427  | early fact present: True
  Answer: Yes, a 15 W payload can be run continuously.  

**Why:**  
- The solar panels provide 40 W in sunlight. With a typical LEO eclipse fraction of 30 %, the average power available over an orbit is 40 W × 0.7 = **28 W**.  
- The continuous load consists of the payload (15 W) plus the OBC (a Raspberry Pi CM4 at 5–7 W, average ~6 W), totaling **~21 W**. Other subsystems (radio at 10 % duty cycle, occasi

=== Strategy: window (API) ===
  Context tokens: 226  | early fact present: False
  Answer: Based on the information provided, it is **not possible to definitively say** whether a 15 W payload can run continuously. The battery sizing answer mentions a "peak draw" and a 60 Wh battery providing ~2 orbits of margin with 30% eclipse, but it does not

## 真实 API 观察

注意 summary 策略中 LLM 摘要保留了 40W 约束 — 好的摘要 prompt 明确告诉模型
「保留数字、预算、决策」。这正是 Claude Code 等 harness 在 auto-compact 时做的事。

## 练习

1. 修改 `build_summary_api` 的摘要 prompt：去掉「preserve hard constraints」指令，重跑，观察最终回答如何退化。
2. 实现两级摘要：每 10 轮把旧摘要 + 新对话再压缩一次。讨论信息损失随轮次的累积。
3. 设计一个「选择性保留」策略：tool 结果只保留成功/失败状态，不保留原始输出。什么场景下这会出错？